# LlamaIndex 快速開始教程

本教程將幫助你快速上手 LlamaIndex，構建第一個 RAG（檢索增強生成）應用。

## 📚 學習目標

- 理解 LlamaIndex 的核心概念
- 安裝和配置 LlamaIndex
- 創建第一個索引
- 執行查詢並獲取響應
- 理解基本的 RAG 工作流程

## 1. 環境設置和安裝

首先安裝必要的套件：

In [ ]:
# 安裝 LlamaIndex 核心套件
!pip install llama-index llama-index-core llama-index-llms-openai llama-index-embeddings-openai -q

## 2. 配置 API Key

LlamaIndex 支持多種 LLM 提供商，這裡我們使用 OpenAI：

In [ ]:
import os
from dotenv import load_dotenv

# 加載環境變數
load_dotenv()

# 設置 OpenAI API Key
# 方法 1: 從 .env 文件加載
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# 方法 2: 直接設置（僅用於測試）
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

print("API Key 已設置！")

## 3. 核心概念介紹

### LlamaIndex 的三個核心組件：

1. **Documents（文檔）**: 你的原始數據
2. **Index（索引）**: 組織和存儲數據的結構
3. **Query Engine（查詢引擎）**: 用於查詢索引的接口

### 工作流程：
```
數據 → 文檔 → 索引 → 查詢引擎 → 響應
```

## 4. 創建示例數據

讓我們創建一些示例文本數據：

In [ ]:
import os

# 創建數據目錄
os.makedirs("data", exist_ok=True)

# 創建示例文件
sample_texts = {
    "ai_intro.txt": """人工智慧（AI）是計算機科學的一個分支，致力於創建能夠執行通常需要人類智慧的任務的系統。
    AI 包括機器學習、深度學習、自然語言處理等多個領域。近年來，大型語言模型（LLM）如 GPT、Claude 等的出現，
    極大地推動了 AI 技術的發展。""",
    
    "machine_learning.txt": """機器學習是人工智慧的一個子領域，專注於開發能夠從數據中學習的算法。
    主要分為監督學習、非監督學習和強化學習三大類。監督學習使用標記數據進行訓練，
    非監督學習從未標記數據中發現模式，強化學習通過與環境互動來學習最優策略。""",
    
    "llm_applications.txt": """大型語言模型（LLM）的應用非常廣泛，包括：
    1. 對話系統和聊天機器人
    2. 文本生成和內容創作
    3. 代碼生成和輔助編程
    4. 文檔問答和知識檢索
    5. 翻譯和文本摘要
    6. 情感分析和文本分類
    RAG（檢索增強生成）技術結合了檢索和生成，能夠提供更準確、更有依據的回答。"""
}

# 寫入文件
for filename, content in sample_texts.items():
    with open(f"data/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

print("示例數據已創建！")
print(f"文件列表: {list(sample_texts.keys())}")

## 5. 加載文檔

使用 `SimpleDirectoryReader` 加載數據目錄中的所有文件：

In [ ]:
from llama_index.core import SimpleDirectoryReader

# 加載文檔
documents = SimpleDirectoryReader("data").load_data()

print(f"加載了 {len(documents)} 個文檔")
print(f"\n第一個文檔的內容預覽：\n{documents[0].text[:200]}...")

## 6. 創建向量索引

`VectorStoreIndex` 是最常用的索引類型，它將文檔轉換為向量嵌入：

In [ ]:
from llama_index.core import VectorStoreIndex

# 創建索引（這一步會調用 OpenAI API 生成嵌入）
print("正在創建索引...")
index = VectorStoreIndex.from_documents(documents)
print("索引創建完成！")

## 7. 創建查詢引擎

查詢引擎用於對索引進行查詢：

In [ ]:
# 創建查詢引擎
query_engine = index.as_query_engine()

print("查詢引擎已準備就緒！")

## 8. 執行查詢

現在我們可以向系統提問了：

In [ ]:
# 查詢 1: 關於 AI 的基本問題
response = query_engine.query("什麼是人工智慧？")
print("問題: 什麼是人工智慧？")
print(f"回答: {response}")
print("\n" + "="*80 + "\n")

In [ ]:
# 查詢 2: 關於機器學習的問題
response = query_engine.query("機器學習有哪些主要類型？")
print("問題: 機器學習有哪些主要類型？")
print(f"回答: {response}")
print("\n" + "="*80 + "\n")

In [ ]:
# 查詢 3: 關於 LLM 應用的問題
response = query_engine.query("LLM 可以應用在哪些場景？")
print("問題: LLM 可以應用在哪些場景？")
print(f"回答: {response}")
print("\n" + "="*80 + "\n")

In [ ]:
# 查詢 4: 關於 RAG 的問題
response = query_engine.query("什麼是 RAG 技術？")
print("問題: 什麼是 RAG 技術？")
print(f"回答: {response}")

## 9. 查看檢索到的原始文檔

我們可以看到 LlamaIndex 從哪些文檔中檢索了信息：

In [ ]:
# 創建帶有詳細信息的查詢引擎
query_engine_detailed = index.as_query_engine(response_mode="tree_summarize")

response = query_engine_detailed.query("解釋一下 RAG 技術")

print("回答:", response)
print("\n" + "="*80)
print("檢索到的源文檔數量:", len(response.source_nodes))
print("\n原始文檔內容:")
for i, node in enumerate(response.source_nodes, 1):
    print(f"\n文檔 {i}:")
    print(f"相似度分數: {node.score:.4f}")
    print(f"內容: {node.text[:200]}...")

## 10. 自定義查詢參數

我們可以調整查詢引擎的參數：

In [ ]:
# 自定義查詢引擎：返回更多相關文檔
query_engine_custom = index.as_query_engine(
    similarity_top_k=5,  # 返回最相似的 5 個文檔塊
    response_mode="compact"  # 使用緊湊模式
)

response = query_engine_custom.query("總結一下關於人工智慧和機器學習的信息")
print(f"回答: {response}")

## 11. 保存和加載索引

我們可以保存索引以避免重複創建：

In [ ]:
# 保存索引
index.storage_context.persist(persist_dir="./storage")
print("索引已保存到 ./storage")

In [ ]:
# 加載索引
from llama_index.core import StorageContext, load_index_from_storage

# 加載存儲的索引
storage_context = StorageContext.from_defaults(persist_dir="./storage")
loaded_index = load_index_from_storage(storage_context)

# 使用加載的索引
loaded_query_engine = loaded_index.as_query_engine()
response = loaded_query_engine.query("什麼是人工智慧？")
print(f"使用加載的索引查詢結果: {response}")

## 12. 使用其他 LLM（可選）

LlamaIndex 支持多種 LLM 提供商：

In [ ]:
# 示例：使用 Google Gemini
# 需要先安裝: pip install llama-index-llms-gemini

from llama_index.core import Settings
from llama_index.llms.gemini import Gemini

# 設置 Gemini API Key
# os.environ["GOOGLE_API_KEY"] = "your-gemini-api-key"

# 使用 Gemini（如果有 API key 的話）
# Settings.llm = Gemini(model="models/gemini-pro")
# gemini_query_engine = index.as_query_engine()
# response = gemini_query_engine.query("什麼是人工智慧？")
# print(f"Gemini 回答: {response}")

print("提示：要使用其他 LLM，請設置相應的 API Key 並取消上面的註釋")

## 📝 總結

在本教程中，我們學習了：

1. ✅ LlamaIndex 的安裝和配置
2. ✅ 核心概念：Documents、Index、Query Engine
3. ✅ 加載文檔數據
4. ✅ 創建向量索引
5. ✅ 執行查詢並獲取響應
6. ✅ 查看檢索的源文檔
7. ✅ 自定義查詢參數
8. ✅ 保存和加載索引
9. ✅ 支持多種 LLM

## 🎯 下一步

- 學習更多數據加載器：`1.數據加載與索引.ipynb`
- 探索不同的查詢引擎：`2.查詢引擎.ipynb`
- 構建聊天系統：`3.Chat_Engine聊天引擎.ipynb`

## 🔗 參考資源

- [LlamaIndex 官方文檔](https://docs.llamaindex.ai/)
- [LlamaIndex GitHub](https://github.com/run-llama/llama_index)
- [Discord 社區](https://discord.gg/dGcwcsnxhU)